1) Ensure both SDKs are available

In [3]:
!pip install -U google-generativeai google-genai


2) Common setup (prompt + helpers)

In [4]:
import os, time, statistics as stats
from typing import Callable, Dict

PROMPT = "Explain transformers in AI in 3 short sentences."
N_RUNS = 3  # repeat to smooth out noise

def summarize_timings(rows):
    ttfb = [r["ttfb_s"] for r in rows if r["ttfb_s"] is not None]
    total = [r["total_s"] for r in rows]
    def f(x): 
        return {"min": min(x), "avg": stats.mean(x), "max": max(x)} if x else None
    return {"TTFB_s": f(ttfb), "Total_s": f(total)}


In [8]:
from dotenv import load_dotenv
load_dotenv("keys.env")

import os
def ok(k): 
    v = os.getenv(k)
    return k, (v is not None and len(v) > 20)

for k, good in map(ok, ["GEMINI_API_KEY","OPENAI_API_KEY","ANTHROPIC_API_KEY"]):
    print(f"{k}: {'OK' if good else 'MISSING'}")


GEMINI_API_KEY: OK
OPENAI_API_KEY: OK
ANTHROPIC_API_KEY: OK


3) “Old” path — google.generativeai (non-streaming)

In [ ]:
# --- OLD SDK: google.generativeai ---
import google.generativeai as genai_old
genai_old.configure(api_key=os.environ["GEMINI_API_KEY"])  # uses your env var

old_model = genai_old.GenerativeModel("models/gemini-1.5-flash")  # or whatever you used before

def run_old_once(prompt=PROMPT, 
                 temperature=0.7, 
                 max_output_tokens=256) -> Dict:
    t0 = time.perf_counter()
    resp = old_model.generate_content(
        prompt,
        generation_config={"temperature": temperature, "max_output_tokens": max_output_tokens},
        safety_settings=None,
    )
    text = resp.text or ""
    t_total = time.perf_counter() - t0
    # Non-streaming: no token-by-token arrival to measure; TTFB is not observable.
    return {"provider": "old(genai v1)", "model": old_model.model_name, "ttfb_s": None, "total_s": t_total, "text": text}


In [18]:
# --- NEW SDK: google.genai ---
from google import genai as genai_new
from google.genai import types

new_client = genai_new.Client()  # uses GEMINI_API_KEY
NEW_MODEL = "gemini-2.5-flash"   # low-latency default

def run_new_once(prompt=PROMPT, 
                 temperature=0.7, 
                 max_output_tokens=256, 
                 thinking_budget=0) -> Dict:
    t0 = time.perf_counter()
    first = None
    out = []

    stream = new_client.models.generate_content_stream(
        model=NEW_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature,
            max_output_tokens=max_output_tokens,
            thinking_config=types.ThinkingConfig(thinking_budget=thinking_budget),
        ),
    )
    for chunk in stream:
        if chunk.text:
            if first is None:
                first = time.perf_counter()
            out.append(chunk.text)
    t_total = time.perf_counter() - t0
    return {
        "provider": "new(genai v2)",
        "model": NEW_MODEL,
        "ttfb_s": None if first is None else first - t0,
        "total_s": t_total,
        "text": "".join(out),
    }


In [19]:
_ = run_old_once(PROMPT)
_ = run_new_once(PROMPT)
print("Warm-up done.")


Warm-up done.


In [20]:
def bench(fn: Callable, label: str):
    rows = []
    for i in range(N_RUNS):
        r = fn(PROMPT)
        rows.append(r)
        print(f"{label} run {i+1}: TTFB={r['ttfb_s']}, Total={r['total_s']:.3f}s")
    return rows

rows_old = bench(run_old_once, "OLD")
rows_new = bench(run_new_once, "NEW")

print("\n--- SUMMARY ---")
print("OLD:", summarize_timings(rows_old))
print("NEW:", summarize_timings(rows_new))


OLD run 1: TTFB=None, Total=0.793s
OLD run 2: TTFB=None, Total=0.854s
OLD run 3: TTFB=None, Total=0.827s
NEW run 1: TTFB=0.6432103999977699, Total=1.085s
NEW run 2: TTFB=0.6589403999969363, Total=1.092s
NEW run 3: TTFB=0.7419961000123294, Total=1.138s

--- SUMMARY ---
OLD: {'TTFB_s': None, 'Total_s': {'min': 0.7934483000135515, 'avg': 0.824975999998666, 'max': 0.8541647999954876}}
NEW: {'TTFB_s': {'min': 0.6432103999977699, 'avg': 0.6813823000023452, 'max': 0.7419961000123294}, 'Total_s': {'min': 1.0854287999973167, 'avg': 1.1051828333341593, 'max': 1.1379286000010325}}


In [21]:
# Try flash-lite + tighter output
from google import genai as genai_new
from google.genai import types
import time, statistics as stats

def run_new_once_model(model, prompt, max_output_tokens=128, temperature=0.5):
    client = genai_new.Client()
    t0 = time.perf_counter()
    first = None
    out = []
    stream = client.models.generate_content_stream(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature,
            max_output_tokens=max_output_tokens,
            thinking_config=types.ThinkingConfig(thinking_budget=0),
            candidate_count=1,  # ensure single candidate
        ),
    )
    for chunk in stream:
        if chunk.text:
            if first is None:
                first = time.perf_counter()
            out.append(chunk.text)
    return {"model": model, "ttfb_s": first - t0 if first else None, "total_s": time.perf_counter() - t0}

def bench_model(model, prompt, n=5):
    # 2x warm-up
    run_new_once_model(model, prompt); run_new_once_model(model, prompt)
    rows = [run_new_once_model(model, prompt) for _ in range(n)]
    ttfb = [r["ttfb_s"] for r in rows if r["ttfb_s"] is not None]
    total = [r["total_s"] for r in rows]
    s = lambda xs: (min(xs), sum(xs)/len(xs), max(xs))
    print(f"\n{model}  (n={n}, max_tokens=128)")
    print("  TTFB  min/avg/max:", s(ttfb))
    print("  Total min/avg/max:", s(total))

PROMPT = "Explain transformers in AI in 3 short sentences."

bench_model("gemini-2.5-flash", PROMPT, n=5)
bench_model("gemini-2.5-flash-lite", PROMPT, n=5)



gemini-2.5-flash  (n=5, max_tokens=128)
  TTFB  min/avg/max: (0.7402667999995174, 0.9060893200017744, 0.9931942999974126)
  Total min/avg/max: (1.1532081000041217, 1.3237670800037449, 1.4228954999998678)

gemini-2.5-flash-lite  (n=5, max_tokens=128)
  TTFB  min/avg/max: (0.5955342999950517, 1.1130471800017403, 1.9890674000052968)
  Total min/avg/max: (0.8525743000063812, 1.346942660000059, 2.2200842000020202)


In [22]:
from google import genai
client = genai.Client()

models = client.models.list()
for m in models:
    print(m.name)


models/embedding-gecko-001
models/gemini-1.5-pro-latest
models/gemini-1.5-pro-002
models/gemini-1.5-pro
models/gemini-1.5-flash-latest
models/gemini-1.5-flash
models/gemini-1.5-flash-002
models/gemini-1.5-flash-8b
models/gemini-1.5-flash-8b-001
models/gemini-1.5-flash-8b-latest
models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-05-20
models/gemini-2.5-flash
models/gemini-2.5-flash-lite-preview-06-17
models/gemini-2.5-pro-preview-05-06
models/gemini-2.5-pro-preview-06-05
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-preview-image-generation
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-2.0-pro-exp
models/gemini-2.0-pro-exp-02-05
models/gemini-exp-1206
models/gemini-2.0-flash-thinking-exp-01-21
models/gemini-2.0-flash-thinking-exp
models/ge

In [27]:
to_test = [
    "gemini-1.5-flash",
    "gemini-1.5-pro",
    "gemini-2.0-flash",
    "gemini-2.0-flash-lite",
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-2.5-pro",
    "gemma-3-4b-it",   # optional, smaller open-weight
]

PROMPT = "Explain transformers in AI in 3 short sentences."
N_RUNS = 5

def bench_models_subset(model_ids):
    results = {}
    for mid in model_ids:
        print(f"\n--- {mid} ---")
        try:
            # warm-up
            run_model(mid, PROMPT)
            times = [run_model(mid, PROMPT) for _ in range(N_RUNS)]
            ttfb = [r["ttfb_s"] for r in times if r["ttfb_s"]]
            total = [r["total_s"] for r in times]
            def stats_summary(xs): return (min(xs), sum(xs)/len(xs), max(xs)) if xs else None
            results[mid] = {"TTFB": stats_summary(ttfb), "Total": stats_summary(total)}
            print("TTFB min/avg/max:", results[mid]["TTFB"])
            print("Total min/avg/max:", results[mid]["Total"])
        except Exception as e:
            print(f"Error running {mid}:", e)
    return results

results = bench_models_subset(to_test)



--- gemini-1.5-flash ---
TTFB min/avg/max: (0.3902649000083329, 0.4323989600001369, 0.49338149999675807)
Total min/avg/max: (0.7725684000033652, 0.8672231600008672, 0.9214599999977509)

--- gemini-1.5-pro ---
Error running gemini-1.5-pro: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-1.5-pro'}}, {'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location

In [26]:
from google import genai
from google.genai import types
import time, statistics as stats

client = genai.Client()

PROMPT = "Explain transformers in AI in 3 short sentences."
N_RUNS = 3

def _needs_thinking(model_id: str) -> bool:
    # 2.5 Pro family requires thinking mode
    return "gemini-2.5-pro" in model_id

def _disallow_thinking(model_id: str) -> bool:
    # Gemma models don't support thinking config
    return model_id.startswith("gemma-")

def run_model(model_id, prompt, max_output_tokens=128, temperature=0.7, retries=2, backoff=2.0):
    """Stream a response and measure TTFB + total with conditional thinking + retries."""
    attempt = 0
    while True:
        try:
            t0 = time.perf_counter()
            first = None
            out = []

            # Build config conditionally
            cfg_kwargs = dict(temperature=temperature, max_output_tokens=max_output_tokens, candidate_count=1)

            if _needs_thinking(model_id):
                cfg_kwargs["thinking_config"] = types.ThinkingConfig(thinking_budget=100)  # small but non-zero
            elif _disallow_thinking(model_id):
                pass  # omit thinking_config entirely
            else:
                cfg_kwargs["thinking_config"] = types.ThinkingConfig(thinking_budget=0)

            stream = client.models.generate_content_stream(
                model=model_id,
                contents=prompt,
                config=types.GenerateContentConfig(**cfg_kwargs),
            )

            for chunk in stream:
                if chunk.text:
                    if first is None:
                        first = time.perf_counter()
                    out.append(chunk.text)

            return {
                "model": model_id,
                "ttfb_s": (first - t0) if first else None,
                "total_s": time.perf_counter() - t0,
                "text": "".join(out),
            }

        except Exception as e:
            msg = str(e)
            transient = ("RESOURCE_EXHAUSTED" in msg) or ("UNAVAILABLE" in msg) or ("429" in msg) or ("503" in msg)
            if transient and attempt < retries:
                attempt += 1
                time.sleep(backoff * attempt)
                continue
            raise

def bench_models_subset(model_ids):
    results = {}
    for mid in model_ids:
        print(f"\n--- {mid} ---")
        try:
            # warm-up (ignore timing)
            _ = run_model(mid, PROMPT)
            # measured runs
            times = [run_model(mid, PROMPT) for _ in range(N_RUNS)]
            ttfb = [r["ttfb_s"] for r in times if r["ttfb_s"]]
            total = [r["total_s"] for r in times]
            def stats_summary(xs): return (min(xs), sum(xs)/len(xs), max(xs)) if xs else None
            results[mid] = {"TTFB": stats_summary(ttfb), "Total": stats_summary(total)}
            print("TTFB min/avg/max:", results[mid]["TTFB"])
            print("Total min/avg/max:", results[mid]["Total"])
        except Exception as e:
            print(f"Error running {mid}:", e)
    return results
